In [3]:
import pandas as pd
import numpy as np
import os

PROCESSED_PATH = "../data/processed"

# Load the order-level dataset
order_level = pd.read_csv(
    os.path.join(PROCESSED_PATH, "order_level.csv")
)

# Convert date columns
order_level["order_purchase_timestamp"] = pd.to_datetime(
    order_level["order_purchase_timestamp"]
)

print("Order-level data loaded successfully.")
print("Shape:", order_level.shape)
print("Columns:")
print(order_level.columns.tolist())

Order-level data loaded successfully.
Shape: (96478, 16)
Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'total_items', 'total_spending_items', 'total_products', 'total_sellers', 'total_payment', 'avg_installments', 'avg_review_score']


In [4]:
# ============================================
# STEP 3: CUSTOMER FEATURE ENGINEERING
# ============================================
# Exploratory customer features are restricted to the observation period.
observation_cutoff = pd.Timestamp("2018-03-01")
observation_order_level = order_level[order_level["order_purchase_timestamp"] < observation_cutoff].copy()
reference_date = observation_order_level["order_purchase_timestamp"].max()
customer_features = (observation_order_level.groupby("customer_unique_id").agg(
    total_orders=("order_id", "nunique"), total_items=("total_items", "sum"),
    total_spending=("total_spending_items", "sum"), avg_order_value=("total_spending_items", "mean"),
    avg_review_score=("avg_review_score", "mean"), total_products=("total_products", "sum"),
    total_sellers=("total_sellers", "sum"), avg_installments=("avg_installments", "mean"),
    first_purchase_date=("order_purchase_timestamp", "min"), last_purchase_date=("order_purchase_timestamp", "max")
).reset_index())
customer_features["recency_days"] = (reference_date - customer_features["last_purchase_date"]).dt.total_seconds() / (60 * 60 * 24)
customer_features["customer_lifetime_days"] = (customer_features["last_purchase_date"] - customer_features["first_purchase_date"]).dt.total_seconds() / (60 * 60 * 24)
print("Observation-only customer features created.")


Customer features created.
Shape: (93358, 13)


,customer_unique_id,total_orders,total_items,total_spending,avg_order_value,avg_review_score,total_products,total_sellers,avg_installments,first_purchase_date,last_purchase_date,recency_days,customer_lifetime_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,129.90,129.90,5.0,1,1,8.0,2018-05-10 10:56:27,2018-05-10 10:56:27,111.169560,0.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,18.90,18.90,4.0,1,1,1.0,2018-05-07 11:11:27,2018-05-07 11:11:27,114.159144,0.0
2,0000f46a3911fa3c0805444483337064,1,1,69.00,69.00,3.0,1,1,8.0,2017-03-10 21:05:03,2017-03-10 21:05:03,536.746921,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,25.99,25.99,4.0,1,1,4.0,2017-10-12 20:29:41,2017-10-12 20:29:41,320.771481,0.0
4,0004aac84e0df4da2b147fca70cf8255,1,1,180.00,180.00,5.0,1,1,6.0,2017-11-14 19:45:42,2017-11-14 19:45:42,287.802025,0.0


In [5]:
customer_features.to_csv(
    os.path.join(PROCESSED_PATH, "customer_features.csv"),
    index=False
)

print("Customer features saved successfully.")

Customer features saved successfully.


In [6]:
print(
    os.path.exists(
        os.path.join(PROCESSED_PATH, "customer_features.csv")
    )
)

True


In [7]:
# ============================================
# STEP 4: TEMPORAL OBSERVATION / FUTURE SPLIT
# ============================================

observation_cutoff = pd.Timestamp("2018-03-01")
future_end = observation_cutoff + pd.Timedelta(days=90)

# Observation period
observation_orders = order_level[
    order_level["order_purchase_timestamp"] < observation_cutoff
].copy()

# Future period
future_orders = order_level[
    (order_level["order_purchase_timestamp"] >= observation_cutoff)
    & (order_level["order_purchase_timestamp"] < future_end)
].copy()

print("Observation orders:", observation_orders.shape)
print("Future orders:", future_orders.shape)

print(
    "\nObservation date range:",
    observation_orders["order_purchase_timestamp"].min(),
    "to",
    observation_orders["order_purchase_timestamp"].max()
)

print(
    "\nFuture date range:",
    future_orders["order_purchase_timestamp"].min(),
    "to",
    future_orders["order_purchase_timestamp"].max()
)

Observation orders: (57319, 16)
Future orders: (39159, 16)

Observation date range: 2016-09-15 12:16:38 to 2018-02-28 23:57:55

Future date range: 2018-03-01 00:00:00 to 2018-08-29 15:00:37


In [8]:
# ============================================
# STEP 5: CUSTOMER FEATURES FROM OBSERVATION
# ============================================

observation_reference_date = observation_orders[
    "order_purchase_timestamp"
].max()

customer_features_observation = (
    observation_orders
    .groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique"),
        total_items=("total_items", "sum"),
        total_spending=("total_spending_items", "sum"),
        avg_order_value=("total_spending_items", "mean"),
        avg_review_score=("avg_review_score", "mean"),
        total_products=("total_products", "sum"),
        total_sellers=("total_sellers", "sum"),
        avg_installments=("avg_installments", "mean"),
        first_purchase_date=("order_purchase_timestamp", "min"),
        last_purchase_date=("order_purchase_timestamp", "max")
    )
    .reset_index()
)

# Recency
customer_features_observation["recency_days"] = (
    observation_reference_date
    - customer_features_observation["last_purchase_date"]
).dt.total_seconds() / (60 * 60 * 24)

# Customer lifetime
customer_features_observation["customer_lifetime_days"] = (
    customer_features_observation["last_purchase_date"]
    - customer_features_observation["first_purchase_date"]
).dt.total_seconds() / (60 * 60 * 24)

print("Observation customer features created.")
print("Shape:", customer_features_observation.shape)

display(customer_features_observation.head())

Observation customer features created.
Shape: (55525, 13)


,customer_unique_id,total_orders,total_items,total_spending,avg_order_value,avg_review_score,total_products,total_sellers,avg_installments,first_purchase_date,last_purchase_date,recency_days,customer_lifetime_days
0,0000f46a3911fa3c0805444483337064,1,1,69.00,69.00,3.0,1,1,8.0,2017-03-10 21:05:03,2017-03-10 21:05:03,355.120046,0.0
1,0000f6ccb0745a6a4b88665a16c9f078,1,1,25.99,25.99,4.0,1,1,4.0,2017-10-12 20:29:41,2017-10-12 20:29:41,139.144606,0.0
2,0004aac84e0df4da2b147fca70cf8255,1,1,180.00,180.00,5.0,1,1,6.0,2017-11-14 19:45:42,2017-11-14 19:45:42,106.175150,0.0
3,00053a61a98854899e70ed204dd4bafe,1,2,382.00,382.00,1.0,2,1,3.0,2018-02-28 11:15:41,2018-02-28 11:15:41,0.529329,0.0
4,0005e1862207bf6ccc02e4228effd9a0,1,1,135.00,135.00,4.0,1,1,3.0,2017-03-04 23:32:12,2017-03-04 23:32:12,361.017859,0.0


In [9]:
# ============================================
# STEP 6: CREATE CHURN LABELS
# ============================================

# Customers who made a purchase in the future period
future_customers = set(
    future_orders["customer_unique_id"].unique()
)

# Customers present during observation period
observation_customers = set(
    customer_features_observation["customer_unique_id"]
)

# Create labels
customer_labels = customer_features_observation[
    ["customer_unique_id"]
].copy()

customer_labels["churn_label"] = (
    ~customer_labels["customer_unique_id"].isin(future_customers)
).astype(int)

print("Customer labels shape:", customer_labels.shape)

print("\nChurn distribution:")
print(customer_labels["churn_label"].value_counts())

print("\nChurn percentage:")
print(
    customer_labels["churn_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Customer labels shape: (55525, 2)

Churn distribution:
churn_label
1    54871
0      654
Name: count, dtype: int64

Churn percentage:
churn_label
1    98.82
0     1.18
Name: proportion, dtype: float64


In [10]:
customer_features_observation.to_csv(
    os.path.join(
        PROCESSED_PATH,
        "customer_features_observation.csv"
    ),
    index=False
)

customer_labels.to_csv(
    os.path.join(
        PROCESSED_PATH,
        "customer_labels.csv"
    ),
    index=False
)

observation_orders.to_csv(
    os.path.join(
        PROCESSED_PATH,
        "observation_orders.csv"
    ),
    index=False
)

future_orders.to_csv(
    os.path.join(
        PROCESSED_PATH,
        "future_orders.csv"
    ),
    index=False
)

print("Temporal datasets saved successfully.")

Temporal datasets saved successfully.
